# Week 10 · Crashing: find the economic optimum

IMSE 456/656 · Program and Project Management · Larson & Gray, Chapter 9

This notebook takes a project network with normal and crash times and costs, shortens it one day at a time by the chapter's least-cost procedure, checks every step against an exact optimisation, and finds the duration where **total cost** (direct plus indirect) is lowest.

**How to use it.** Choose File → Save a copy in Drive, then Runtime → Run all. Edit only the cell marked **EDIT**. By default the notebook downloads the same conference workbook you import into Smartsheet.

| Workbook column | Meaning |
|:--|:--|
| `Activity`, `ID` | The activity and its label |
| `Predecessors` | Smartsheet row numbers of the immediate predecessors, such as `4, 5` (letter IDs also work) |
| `Normal days`, `Normal cost` | The low-cost way to do the activity |
| `Crash days`, `Crash cost` | The fastest realistic way, and what it costs |

Assumptions, as in the chapter: cost rises in a straight line from the normal point to the crash point, an activity can be shortened one day at a time, and indirect cost is a fixed amount per day of project duration.

In [ ]:
# ---------------------------------------------------------------- EDIT
DATA_SOURCE = "course"     # "course", "upload", "table", or a file path
COURSE_FILE = "https://raw.githubusercontent.com/harunpirim/IMSE456/main/files/week-10-crashing-workbook.xlsx"

INDIRECT_PER_DAY = 1000    # dollars of indirect cost per workday of project duration
INDIRECT_OFFSET = 0        # added to every duration's indirect cost; leave at 0
START_DATE = "2026-11-02"  # project start, for calendar dates (Monday to Friday, no holidays)

# Used when DATA_SOURCE = "table", and as the fallback if the download fails
#   id,  activity,                     predecessors, normal days, normal cost, crash days, crash cost
TABLE = [
    ("A", "Approve event brief",        [],          2,  2000, 1,  3200),
    ("B", "Contract venue",             ["A"],       5, 30000, 3, 31400),
    ("C", "Draft program",              ["A"],       6,  6000, 4,  6800),
    ("D", "Configure registration",     ["B"],       4,  8000, 3,  8900),
    ("E", "Confirm speakers",           ["C"],       8, 12000, 5, 13800),
    ("F", "Recruit exhibitors",         ["C"],       5,  2500, 3,  3100),
    ("G", "Publish program",            ["D", "E"],  3,  3000, 2,  4500),
    ("H", "Prepare on-site materials",  ["F", "G"],  4,  4000, 2,  5600),
    ("I", "Complete rehearsal",         ["H"],       2,  2500, 2,  2500),
]

## 0 · Load the activity table

Check that the table printed below matches the workbook before going on.

In [ ]:
import io, urllib.request, itertools
import numpy as np, pandas as pd

def from_excel(src):
    df = pd.read_excel(src, sheet_name=0); df.columns = [str(c).strip() for c in df.columns]
    need = ["Activity", "ID", "Predecessors", "Normal days", "Normal cost", "Crash days", "Crash cost"]
    missing = [c for c in need if c not in df.columns]
    assert not missing, f"the workbook is missing column(s) {missing}; it has {list(df.columns)}"
    df = df[df["ID"].notna()].reset_index(drop=True); ids = [str(x).strip() for x in df["ID"]]; rows = []
    for k, r in df.iterrows():
        pred = []
        if pd.notna(r["Predecessors"]):
            for tok in str(r["Predecessors"]).replace(";", ",").split(","):
                tok = tok.strip()
                if not tok: continue
                if tok in ids: pred.append(tok)
                else:
                    n = int(float(tok)); assert 1 <= n <= len(ids), f"row {k + 1}: predecessor row {n} does not exist"
                    pred.append(ids[n - 1])
        rows.append((ids[k], str(r["Activity"]), pred, int(r["Normal days"]), float(r["Normal cost"]), int(r["Crash days"]), float(r["Crash cost"])))
    return rows

if DATA_SOURCE == "course":
    try:
        TABLE = from_excel(io.BytesIO(urllib.request.urlopen(COURSE_FILE, timeout=30).read())); print("Loaded the conference workbook from the course repository.")
    except Exception as e:
        print(f"Could not download the course workbook ({e}); using TABLE from the EDIT cell.")
elif DATA_SOURCE == "upload":
    from google.colab import files
    up = files.upload(); name = next(iter(up)); TABLE = from_excel(io.BytesIO(up[name])); print(f"Loaded {name}.")
elif DATA_SOURCE != "table":
    TABLE = from_excel(DATA_SOURCE); print(f"Loaded {DATA_SOURCE}.")
else:
    print("Using TABLE from the EDIT cell.")

ACT = {a: dict(name=nm, pred=p, n=n, nc=nc, c=c, cc=cc) for a, nm, p, n, nc, c, cc in TABLE}
IDS = list(ACT)
for a, x in ACT.items():
    assert 0 < x["c"] <= x["n"], f"{a}: crash days must be between 1 and normal days"
    assert x["cc"] >= x["nc"], f"{a}: crash cost must be at least normal cost"
    for p in x["pred"]: assert p in ACT, f"{a}: predecessor {p} is not an activity"
SLOPE = {a: (x["cc"] - x["nc"]) / (x["n"] - x["c"]) if x["n"] > x["c"] else None for a, x in ACT.items()}
pd.DataFrame({"activity": [ACT[a]["name"] for a in IDS], "predecessors": [", ".join(ACT[a]["pred"]) or "—" for a in IDS],
              "normal days": [ACT[a]["n"] for a in IDS], "crash days": [ACT[a]["c"] for a in IDS],
              "normal cost": [ACT[a]["nc"] for a in IDS], "crash cost": [ACT[a]["cc"] for a in IDS],
              "cost per day saved": [SLOPE[a] if SLOPE[a] is not None else "cannot crash" for a in IDS],
              "days that can be saved": [ACT[a]["n"] - ACT[a]["c"] for a in IDS]}, index=IDS)

## 1 · The normal schedule and its paths

Every complete path through the network, with its length at normal times. The longest is the critical path, and only shortening critical activities shortens the project.

In [ ]:
SUCC = {a: [s for s in IDS if a in ACT[s]["pred"]] for a in IDS}
def all_paths():
    out = []
    def rec(a, pth):
        if len(out) > 20000: raise RuntimeError("more than 20,000 paths; use the exact method only")
        if not SUCC[a]: out.append(pth + [a]); return
        for s in SUCC[a]: rec(s, pth + [a])
    for a in IDS:
        if not ACT[a]["pred"]: rec(a, [])
    return out
PATHS = all_paths()
length = lambda p, d: sum(d[a] for a in p)
NORMAL = {a: ACT[a]["n"] for a in IDS}
T_NORMAL = max(length(p, NORMAL) for p in PATHS)
indirect = lambda T: INDIRECT_PER_DAY * T + INDIRECT_OFFSET
def finish_date(T): return np.busday_offset(np.datetime64(START_DATE), T - 1, roll="forward")
pd.DataFrame({"path": ["–".join(p) for p in PATHS], "length at normal times": [length(p, NORMAL) for p in PATHS]}).sort_values(
    "length at normal times", ascending=False).reset_index(drop=True)

In [ ]:
print(f"Normal duration: {T_NORMAL} workdays, finishing {pd.Timestamp(finish_date(T_NORMAL)):%a %d %b %Y}")
print(f"Normal direct cost ${sum(x['nc'] for x in ACT.values()):,.0f} + indirect ${indirect(T_NORMAL):,.0f} = ${sum(x['nc'] for x in ACT.values()) + indirect(T_NORMAL):,.0f}")

## 2 · The chapter's procedure, one day at a time

At each step: find the critical paths; among activities that can still be shortened, choose the **cheapest set** that shortens every critical path by one day (one shared activity, or one activity on each path); shorten them by a day; repeat until no critical path can be shortened. The procedure never undoes an earlier crash.

In [ ]:
def chapter_procedure():
    red = {a: 0 for a in IDS}; direct = sum(ACT[a]["nc"] for a in IDS); rows = []
    d = {a: ACT[a]["n"] for a in IDS}; T = max(length(p, d) for p in PATHS)
    rows.append(dict(days=T, shortened="—", step_cost=0.0, direct=direct, critical=", ".join(sorted({a for p in PATHS if length(p, d) == T for a in p}))))
    while True:
        d = {a: ACT[a]["n"] - red[a] for a in IDS}; T = max(length(p, d) for p in PATHS)
        crit = [p for p in PATHS if length(p, d) == T]
        cand = [a for a in IDS if SLOPE[a] is not None and red[a] < ACT[a]["n"] - ACT[a]["c"] and any(a in p for p in crit)]
        best = None
        for k in range(1, len(cand) + 1):
            for combo in itertools.combinations(cand, k):
                if all(any(a in p for a in combo) for p in crit):
                    cost = sum(SLOPE[a] for a in combo)
                    if best is None or cost < best[0] - 1e-9: best = (cost, combo)
            if best is not None and k >= len(crit): break
        if best is None: break
        for a in best[1]: red[a] += 1
        direct += best[0]; d = {a: ACT[a]["n"] - red[a] for a in IDS}; T2 = max(length(p, d) for p in PATHS)
        rows.append(dict(days=T2, shortened=" + ".join(best[1]), step_cost=best[0], direct=direct,
                         critical=", ".join(sorted({a for p in PATHS if length(p, d) == T2 for a in p}))))
    return pd.DataFrame(rows), red

STEPS, RED_MIN = chapter_procedure()
STEPS["indirect"] = STEPS["days"].map(indirect); STEPS["total"] = STEPS["direct"] + STEPS["indirect"]
STEPS["finish"] = STEPS["days"].map(lambda T: f"{pd.Timestamp(finish_date(T)):%a %d %b}")
STEPS.style.format({"step_cost": "${:,.0f}", "direct": "${:,.0f}", "indirect": "${:,.0f}", "total": "${:,.0f}"}).hide(axis="index")

## 3 · Check every step exactly

The chapter's procedure is a rule. For each duration, a linear program finds the cheapest possible way to reach it, allowing any combination of shortenings (Kelley, 1961). If the two agree at every duration, the procedure found the true cost curve for this project.

In [ ]:
from scipy.optimize import linprog
def exact_direct(T):
    n = len(IDS); idx = {a: i for i, a in enumerate(IDS)}
    # variables: start time s_a (n), then reduction y_a (n)
    c = [0.0] * n + [SLOPE[a] or 0.0 for a in IDS]
    A, b = [], []
    for a in IDS:
        for s in SUCC[a]:  # s_a + n_a - y_a <= s_s
            row = [0.0] * 2 * n; row[idx[a]] = 1; row[idx[s]] = -1; row[n + idx[a]] = -1; A.append(row); b.append(-ACT[a]["n"])
        row = [0.0] * 2 * n; row[idx[a]] = 1; row[n + idx[a]] = -1; A.append(row); b.append(T - ACT[a]["n"])  # finish by T
    bounds = [(0, None)] * n + [(0, ACT[a]["n"] - ACT[a]["c"]) for a in IDS]
    r = linprog(c, A_ub=A, b_ub=b, bounds=bounds, method="highs")
    return (sum(ACT[a]["nc"] for a in IDS) + r.fun) if r.status == 0 else None

STEPS["exact direct"] = STEPS["days"].map(exact_direct)
STEPS["agrees"] = (STEPS["direct"] - STEPS["exact direct"]).abs() < 0.5
print("The chapter procedure matches the exact optimum at every duration." if STEPS["agrees"].all()
      else "The procedure is more expensive than the exact optimum at: " + ", ".join(str(t) for t in STEPS.loc[~STEPS["agrees"], "days"]))
STEPS[["days", "direct", "exact direct", "agrees"]].style.format({"direct": "${:,.0f}", "exact direct": "${:,.0f}"}).hide(axis="index")

## 4 · The cost-duration graph and the economic optimum

Total cost is direct plus indirect. Crashing pays while the cost of the next day saved is less than the indirect cost of a day. The optimum is the lowest total.

In [ ]:
import matplotlib.pyplot as plt
best = STEPS.loc[STEPS["total"].idxmin()]
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6.5), sharex=True, gridspec_kw={"height_ratios": [1, 1.2]})
ax1.plot(STEPS["days"], STEPS["total"], "-o", color="#0F5A4B", lw=2.5)
ax1.plot([best["days"]], [best["total"]], "o", ms=13, mfc="white", mec="#0F5A4B", mew=2.5)
ax1.annotate(f"Optimum: {int(best['days'])} days, ${best['total']:,.0f}", (best["days"], best["total"]), textcoords="offset points", xytext=(0, -26), ha="center", color="#0F5A4B", fontweight="bold")
ax1.set_ylabel("total cost, $"); ax1.set_title("Total cost (zoomed)", loc="left", color="#0F5A4B", fontweight="bold")
ax2.plot(STEPS["days"], STEPS["direct"], "--o", color="#627571", ms=4, label="direct")
ax2.plot(STEPS["days"], STEPS["indirect"], ":o", color="#4E8A7C", ms=4, label="indirect")
ax2.set_ylim(0, None); ax2.set_ylabel("$"); ax2.set_xlabel("project duration, workdays"); ax2.legend(frameon=False)
for ax in (ax1, ax2): ax.spines[["top", "right"]].set_visible(False); ax.invert_xaxis()
plt.tight_layout(); plt.show()
print(f"Economic optimum: {int(best['days'])} workdays, finishing {best['finish']}, total ${best['total']:,.0f}")
print(f"Shortest possible: {int(STEPS['days'].min())} workdays, total ${STEPS['total'].iloc[-1]:,.0f}")

## 5 · What crashing everything would cost

Shorten every activity to its crash time, needed or not, and compare.

In [ ]:
crash_all = {a: ACT[a]["c"] for a in IDS}
T_all = max(length(p, crash_all) for p in PATHS); direct_all = sum(ACT[a]["cc"] for a in IDS)
wasted = [a for a in IDS if SLOPE[a] is not None and ACT[a]["n"] - ACT[a]["c"] > RED_MIN[a]]
print(f"Crash everything: {T_all} workdays, direct ${direct_all:,.0f} + indirect ${indirect(T_all):,.0f} = ${direct_all + indirect(T_all):,.0f}")
print(f"Normal plan:      {T_NORMAL} workdays, total ${STEPS['total'].iloc[0]:,.0f}")
print(f"Economic optimum: {int(best['days'])} workdays, total ${best['total']:,.0f}")
print("Crashing these bought no time at all: " + ", ".join(f"{a} ({ACT[a]['n'] - ACT[a]['c'] - RED_MIN[a]} day(s), ${(ACT[a]['n'] - ACT[a]['c'] - RED_MIN[a]) * SLOPE[a]:,.0f})" for a in wasted))

## 6 · Try it

- **Indirect cost changes.** Set `INDIRECT_PER_DAY = 700` (the venue hold is released) and run all. Then try `1300`. Where does the optimum move, and which activity's cost per day explains each move?
- **The chapter's example.** Set `DATA_SOURCE = "table"` and replace `TABLE` with Figure 9.3's data: A (none, 3, $50, 2, $70); B (A, 6, $80, 4, $160); C (A, 10, $60, 9, $90); D (A, 11, $50, 7, $150); E (B, 8, $100, 6, $160); F (C and D, 5, $40, 4, $70); G (E and F, 6, $70, 6, $70). Set `INDIRECT_PER_DAY = 50` and `INDIRECT_OFFSET = -850`, which gives the chapter's $400 at 25 units falling $50 a unit. The optimum should be 22 units at $775.
- **Your own project.** Fill in a copy of the workbook and set `DATA_SOURCE = "upload"`.

Reference: Kelley, J. E. (1961). Critical-path planning and scheduling: Mathematical basis. *Operations Research*, 9(3), 296–320.